In [1]:
#Patrones y factores asociados a la consumación de extorsiones registradas por Seguridad 
#Ciudadana en Nezahualcóyotl mediante minería de datos

In [2]:
#Bloque 0. Instalación de librerías
#Ejecuta esto una sola vez
!pip install pandas numpy openpyxl scikit-learn matplotlib mlxtend kmodes


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
#Bloque 1. Cargar librerías y archivo
#Fase KDD: Selección de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import unicodedata

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

from kmodes.kmodes import KModes


ruta = "extorsiones.xlsx"
df = pd.read_excel(ruta)

print("Dimensiones:", df.shape)
print("\nColumnas originales:")
print(df.columns.tolist())

df.head()

Dimensiones: (1189, 13)

Columnas originales:
['FECHA', 'HORA', 'CUADRANTE', 'DELITO ', 'Delito2a parte', 'OBJETOS ROBADOS', 'UNIDAD QUE PRESTA APOYO', 'DETENCION (PUESTA/REMISION)', 'JERARQUIA', 'SEXO VICTIMA', ' EDAD', 'RANGO DE EDAD', 'GIRO DEL COMERCIO']


,FECHA,HORA,CUADRANTE,DELITO,Delito2a parte,OBJETOS ROBADOS,UNIDAD QUE PRESTA APOYO,DETENCION (PUESTA/REMISION),JERARQUIA,SEXO VICTIMA,EDAD,RANGO DE EDAD,GIRO DEL COMERCIO
0,2025-04-01,18:26:00,C016,LLAMADA DE EXTORSIÓN,Consumada,"$3,000.00 PESOS",V016,DENUNCIA,ALTO IMPACTO,F,74,56 Y MAS,NO APLICA
1,2025-04-01,10:53:00,C062,LLAMADA DE EXTORSIÓN,Evitada,NO APLICA,V062,DENUNCIA,BAJO IMPACTO,F,41,36-45 AÑOS,LOCAL DE COMIDA
2,2025-04-01,12:00:00,C066,LLAMADA DE EXTORSIÓN,Evitada,NO APLICA,V061,DENUNCIA,BAJO IMPACTO,F,36,36-45 AÑOS,LOCAL DE COMIDA
3,2025-04-01,10:02:00,C076,LLAMADA DE EXTORSIÓN,Evitada,NO APLICA,V076,DENUNCIA,BAJO IMPACTO,M,45,36-45 AÑOS,LOCAL DE COMIDA
4,2025-04-02,13:35:00,C014,MENSAJES DE EXTORSIÓN,Evitada,NO APLICA,CASETA 1A,DENUNCIA,BAJO IMPACTO,F,27,26-35 AÑOS,NO APLICA


In [5]:
# BLOQUE 2 Normalizar nombres de columnas
#Aquí está la corrección clave
#Este bloque sustituye el renombrado manual.
#Así evitamos errores por espacios, acentos o paréntesis.
#Fase KDD: Selección + preparación inicial

def normalizar_nombre_columna(col):
    col = str(col).strip().upper()
    col = ''.join(
        c for c in unicodedata.normalize('NFD', col)
        if unicodedata.category(c) != 'Mn'
    )
    col = re.sub(r'[^A-Z0-9]+', '_', col)
    col = re.sub(r'_+', '_', col).strip('_')
    return col

df.columns = [normalizar_nombre_columna(c) for c in df.columns]

print("Columnas normalizadas:")
print(df.columns.tolist())

Columnas normalizadas:
['FECHA', 'HORA', 'CUADRANTE', 'DELITO', 'DELITO2A_PARTE', 'OBJETOS_ROBADOS', 'UNIDAD_QUE_PRESTA_APOYO', 'DETENCION_PUESTA_REMISION', 'JERARQUIA', 'SEXO_VICTIMA', 'EDAD', 'RANGO_DE_EDAD', 'GIRO_DEL_COMERCIO']


In [6]:
#Bloque 3. Auditoría rápida de calidad
#Fase KDD: Preprocesamiento
#Esto te dice cuántos valores únicos tiene cada variable, cuántos nulos hay y ejemplos de categorías.

print("Nulos por columna:\n")
print(df.isna().sum())

print("\nValores únicos por columna:\n")
for col in df.columns:
    print(f"\n--- {col} ---")
    print("n_unique:", df[col].nunique(dropna=False))
    print(df[col].astype(str).value_counts(dropna=False).head(10))

Nulos por columna:

FECHA                        0
HORA                         0
CUADRANTE                    0
DELITO                       0
DELITO2A_PARTE               0
OBJETOS_ROBADOS              0
UNIDAD_QUE_PRESTA_APOYO      0
DETENCION_PUESTA_REMISION    0
JERARQUIA                    0
SEXO_VICTIMA                 0
EDAD                         5
RANGO_DE_EDAD                0
GIRO_DEL_COMERCIO            0
dtype: int64

Valores únicos por columna:


--- FECHA ---
n_unique: 306
FECHA
2025-08-25    13
2025-06-12    10
2025-07-11    10
2025-09-26    10
2025-10-08    10
2025-07-09     9
2025-07-21     9
2025-07-24     9
2025-08-08     9
2025-08-26     9
Name: count, dtype: int64

--- HORA ---
n_unique: 442
HORA
17:00:00    23
14:00:00    23
14:30:00    20
12:00:00    19
15:00:00    18
12:30:00    17
16:00:00    17
16:30:00    16
11:30:00    16
13:00:00    15
Name: count, dtype: int64

--- CUADRANTE ---
n_unique: 107
CUADRANTE
C131       29
C088       22
C087       22
C064     

In [7]:
#Bloque 4. Funciones de limpieza
#Fase KDD: Preprocesamiento
#Aquí limpiamos texto, acentos, espacios y estandarizamos categorías.

def limpiar_texto(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    x = re.sub(r"\s+", " ", x)
    return x

def quitar_acentos(texto):
    if pd.isna(texto):
        return np.nan
    texto = str(texto)
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto

cols_texto_esperadas = [
    "CUADRANTE",
    "DELITO",
    "DELITO2A_PARTE",
    "OBJETOS_ROBADOS",
    "UNIDAD_QUE_PRESTA_APOYO",
    "DETENCION_PUESTA_REMISION",
    "JERARQUIA",
    "SEXO_VICTIMA",
    "RANGO_DE_EDAD",
    "GIRO_DEL_COMERCIO"
]

# Solo usa columnas que realmente existen
cols_texto = [c for c in cols_texto_esperadas if c in df.columns]

for col in cols_texto:
    df[col] = df[col].apply(limpiar_texto)

# Estandarizaciones puntuales
if "JERARQUIA" in df.columns:
    df["JERARQUIA"] = df["JERARQUIA"].replace({
        "ALTO IMPACTO ": "ALTO IMPACTO",
        "ALTO  IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO.": "ALTO IMPACTO",
        "ALTO IMPACTO,": "ALTO IMPACTO",
        "ALTO IMPACTO/": "ALTO IMPACTO",
        "ALTO IMPACTO-": "ALTO IMPACTO",
        "ALTO IMPACTO_": "ALTO IMPACTO",
        "ALTO IMPACTO  ": "ALTO IMPACTO",
        "ALTO IMPACTO\t": "ALTO IMPACTO",
        "ALTO IMPACTO\r": "ALTO IMPACTO",
        "ALTO IMPACTO\n": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO".upper(): "ALTO IMPACTO",
        "ALTO IMPACTO".strip(): "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "ALTO IMPACTO": "ALTO IMPACTO",
        "AlTO IMPACTO".upper(): "ALTO IMPACTO"
    })

if "SEXO_VICTIMA" in df.columns:
    df["SEXO_VICTIMA"] = df["SEXO_VICTIMA"].replace({
        "F ": "F",
        "M ": "M",
        "NO ESPECIFICA": "NP"
    })

if "RANGO_DE_EDAD" in df.columns:
    df["RANGO_DE_EDAD"] = df["RANGO_DE_EDAD"].replace({
        "56 Y MAS ": "56 Y MAS"
    })

print("\nValores de SEXO_VICTIMA:")
print(df["SEXO_VICTIMA"].value_counts(dropna=False))

print("\nValores de JERARQUIA:")
print(df["JERARQUIA"].value_counts(dropna=False))

print("\nValores de RANGO_DE_EDAD:")
print(df["RANGO_DE_EDAD"].value_counts(dropna=False))


Valores de SEXO_VICTIMA:
SEXO_VICTIMA
M        632
F        544
F Y M      9
NP         4
Name: count, dtype: int64

Valores de JERARQUIA:
JERARQUIA
BAJO IMPACTO    1056
ALTO IMPACTO     133
Name: count, dtype: int64

Valores de RANGO_DE_EDAD:
RANGO_DE_EDAD
56 Y MAS       253
26-35 AÑOS     248
46-55 AÑOS     219
36-45 AÑOS     216
18-25 AÑOS     116
DESCONOCIDO    110
02-17 AÑOS      27
Name: count, dtype: int64


In [8]:
#BLOQUE 5. Limpieza de fecha, hora y edad
#Fase KDD: Preprocesamiento

In [9]:
# FECHA
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")

# HORA
def convertir_hora(x):
    if pd.isna(x):
        return np.nan
    try:
        return pd.to_datetime(str(x), format="%H:%M:%S", errors="coerce").time()
    except:
        try:
            return pd.to_datetime(str(x), format="%H:%M", errors="coerce").time()
        except:
            return np.nan

df["HORA"] = df["HORA"].apply(convertir_hora)

# EDAD
df["EDAD"] = df["EDAD"].astype(str).str.strip().replace({
    "NP": np.nan,
    "N/P": np.nan,
    "NO ESPECIFICA": np.nan,
    "NAN": np.nan
})

df["EDAD_LIMPIA"] = pd.to_numeric(df["EDAD"], errors="coerce")

print(df[["FECHA", "HORA", "EDAD", "EDAD_LIMPIA"]].head())
print("\nNulos en EDAD_LIMPIA:", df["EDAD_LIMPIA"].isna().sum())

       FECHA      HORA EDAD  EDAD_LIMPIA
0 2025-04-01  18:26:00   74         74.0
1 2025-04-01  10:53:00   41         41.0
2 2025-04-01  12:00:00   36         36.0
3 2025-04-01  10:02:00   45         45.0
4 2025-04-02  13:35:00   27         27.0

Nulos en EDAD_LIMPIA: 115


In [11]:
#BLOQUE 6. Variables derivadas
#Fase KDD: Transformación

In [12]:
# Variable objetivo binaria
df["TARGET_CONSUMADA"] = df["DELITO2A_PARTE"].map({
    "EVITADA": 0,
    "CONSUMADA": 1
})

# Mes
df["MES"] = df["FECHA"].dt.month

# Día de semana
dias = {
    0: "LUNES",
    1: "MARTES",
    2: "MIERCOLES",
    3: "JUEVES",
    4: "VIERNES",
    5: "SABADO",
    6: "DOMINGO"
}
df["DIA_SEMANA"] = df["FECHA"].dt.dayofweek.map(dias)

# Fin de semana
df["FIN_SEMANA"] = df["DIA_SEMANA"].isin(["SABADO", "DOMINGO"]).astype(int)

# Convertir hora a número
def hora_a_numero(hora):
    if pd.isna(hora):
        return np.nan
    return hora.hour + hora.minute / 60

df["HORA_NUM"] = df["HORA"].apply(hora_a_numero)

# Franja horaria
def clasificar_franja(h):
    if pd.isna(h):
        return "DESCONOCIDA"
    if 0 <= h < 6:
        return "MADRUGADA"
    elif 6 <= h < 12:
        return "MANANA"
    elif 12 <= h < 18:
        return "TARDE"
    else:
        return "NOCHE"

df["FRANJA_HORARIA"] = df["HORA_NUM"].apply(clasificar_franja)

# Tipo general de extorsión
def tipo_extorsion_general(delito):
    if pd.isna(delito):
        return "OTRO"

    d = quitar_acentos(str(delito).upper())

    if "LLAMADA" in d:
        return "LLAMADA"
    elif "MENSAJE" in d:
        return "MENSAJE"
    elif "PRESENCIAL" in d:
        return "PRESENCIAL"
    elif "EXTORSION" in d:
        return "EXTORSION"
    else:
        return "OTRO"

df["TIPO_EXTORSION"] = df["DELITO"].apply(tipo_extorsion_general)

# Agrupar giros poco frecuentes
top_giros = df["GIRO_DEL_COMERCIO"].value_counts().head(15).index
df["GIRO_COMERCIO_RED"] = np.where(
    df["GIRO_DEL_COMERCIO"].isin(top_giros),
    df["GIRO_DEL_COMERCIO"],
    "OTROS"
)

print(df[[
    "DELITO2A_PARTE",
    "TARGET_CONSUMADA",
    "MES",
    "DIA_SEMANA",
    "FIN_SEMANA",
    "HORA_NUM",
    "FRANJA_HORARIA",
    "TIPO_EXTORSION",
    "GIRO_COMERCIO_RED"
]].head())

  DELITO2A_PARTE  TARGET_CONSUMADA  MES DIA_SEMANA  FIN_SEMANA   HORA_NUM  \
0      CONSUMADA                 1    4     MARTES           0  18.433333   
1        EVITADA                 0    4     MARTES           0  10.883333   
2        EVITADA                 0    4     MARTES           0  12.000000   
3        EVITADA                 0    4     MARTES           0  10.033333   
4        EVITADA                 0    4  MIERCOLES           0  13.583333   

  FRANJA_HORARIA TIPO_EXTORSION GIRO_COMERCIO_RED  
0          NOCHE        LLAMADA         NO APLICA  
1         MANANA        LLAMADA   LOCAL DE COMIDA  
2          TARDE        LLAMADA   LOCAL DE COMIDA  
3         MANANA        LLAMADA   LOCAL DE COMIDA  
4          TARDE        MENSAJE         NO APLICA  


In [13]:
#BLOQUE 7. Guardar base limpia
#Fase KDD: Cierre del preprocesamiento

In [14]:
df.to_excel("extorsiones_limpia.xlsx", index=False)
print("Archivo guardado: extorsiones_limpia.xlsx")

Archivo guardado: extorsiones_limpia.xlsx
